# Liquidation Strategy Walkthrough

This walkthrough follows the same service layer used by the Streamlit application for Liquidity Management Tools Calibration. It is written for fund-risk review: the code cells call application services, while the markdown explains the business meaning of the returned results.

The walkthrough covers both the single-period calibration workflow and the implemented 12-month redemption-path workflow. It does not reimplement liquidation, market stress, redemption demand, LMT assessment, or path calculations inside notebook cells.

## Setup

The setup imports display helpers, pandas, and the application service functions. The calculation boundary is `lmt_calibration.services`, matching the app workflow.

In [ ]:
from decimal import Decimal

import pandas as pd
from _notebook_helpers import days, gross_sales, money, print_profile, rate, yes_no
from _notebook_setup import SAMPLE_DATA_DIR, configure_display

from lmt_calibration.services import (
    build_historical_result_rows,
    build_scenario_matrix_outcome,
    build_t0_liquidity_profile_rows,
    fund_positions,
    load_app_sample_data,
    run_sample_redemption_path,
    run_scenario_across_market_conditions,
    run_selected_sample_scenario,
)

configure_display()

## Load Application Sample Data

The sample files are loaded through the same application service used by Streamlit. This keeps the walkthrough aligned with loader validation, domain-object construction, and service-level lookups.

In [ ]:
inputs = load_app_sample_data(SAMPLE_DATA_DIR)

sample_input_counts = [
    ("funds", len(inputs.funds)),
    ("positions", len(inputs.positions)),
    ("investor_classes", len(inputs.investor_classes)),
    ("redemption_scenarios", len(inputs.redemption_scenarios)),
    ("market_stresses", len(inputs.market_stresses)),
    ("liquidity_stresses", len(inputs.liquidity_stresses)),
    ("scenario_definitions", len(inputs.scenario_definitions)),
    ("lmt_parameters", len(inputs.lmt_parameters)),
    ("liquidation_strategies", len(inputs.liquidation_strategies)),
]

pd.DataFrame(sample_input_counts, columns=["dataset", "records"])

# Part 1 - Single-Period Application Workflow

The single-period workflow is the current calibration matrix workflow. A selected fund, redemption scenario, market stress, liquidity stress, liquidation strategy, and LMT parameter set are assembled by the service layer. The service applies the methodology and returns a run object containing the inputs, liquidation result, execution-cost context, and simulated activation assessment.

In [ ]:
selected_fund_id = "lux_dynamic_allocation"
selected_redemption_scenario_id = "severe_platform_outflow"
selected_strategy_id = "partial_cash_then_pro_rata"

single_period_run = run_selected_sample_scenario(
    inputs,
    fund_id=selected_fund_id,
    strategy_id=selected_strategy_id,
    redemption_scenario_id_override=selected_redemption_scenario_id,
)
single_period_result = single_period_run.result

scenario_profile = {
    "fund_id": single_period_run.fund.fund_id,
    "fund_name": single_period_run.fund.fund_name,
    "redemption_scenario": single_period_run.redemption.name,
    "market_stress": single_period_run.market_stress.name,
    "liquidity_stress": single_period_run.liquidity_stress.name,
    "liquidation_strategy": single_period_run.strategy.name,
    "strategy_type": single_period_run.strategy.strategy_type.value,
    "lmt_parameter_set": single_period_run.parameters.parameter_set_id,
}

print_profile(scenario_profile)

## Fund And Portfolio Snapshot

The fund snapshot and position set are the opening point for the application workflow. Cash, listed equities, listed ETFs, reverse repos, and repo financing exposures are intentionally shown separately because the liquidation engine treats their liquidity differently.

In [ ]:
fund = single_period_run.fund
positions = fund_positions(inputs, fund)
liquidity_profile_rows = build_t0_liquidity_profile_rows(positions)

fund_profile = {
    "as_of_date": str(fund.as_of_date),
    "base_currency": fund.base_currency,
    "nav": money(fund.nav),
    "dealing_frequency": fund.dealing_frequency,
    "redemption_notice_days": days(fund.redemption_notice_days),
    "redemption_settlement_days": days(fund.redemption_settlement_days),
}

print_profile(fund_profile)

In [ ]:
pd.DataFrame(
    [
        {
            "instrument_name": position.instrument_name,
            "asset_group": position.asset_group.value,
            "market_value": money(position.market_value),
            "notional_amount": money(position.notional_amount),
            "base_haircut_rate": rate(position.base_haircut_rate),
            "base_liquidity_capacity_rate": rate(position.base_liquidity_capacity_rate),
            "settlement_days": days(position.settlement_days),
            "maturity_days": days(position.maturity_days),
        }
        for position in positions
    ]
)

## Liquidity Profile At The Opening Date

The service prepares the same liquidity-bucket rows used by the application. The rows separate NAV by liquidity bucket and estimate liquid resources available under the current position attributes.

In [ ]:
pd.DataFrame(
    [
        {
            "liquidity_bucket": row["liquidity_bucket"],
            "nav_amount": money(row["nav_amount"]),
            "liquid_resources": money(row["liquid_resources"]),
            "unavailable_nav": money(row["unavailable_nav"]),
        }
        for row in liquidity_profile_rows
    ]
)

## Redemption Pressure And Liquidation Result

The service computes the redemption amount from investor-class assumptions and the selected redemption scenario. The liquidation result then shows how the selected strategy uses cash and asset sales to meet that cash need subject to the minimum buffer, capacity, haircut, settlement, and maturity constraints.

In [ ]:
liquidation_summary = {
    "total_redemption_amount": money(single_period_result.total_redemption_amount),
    "redemption_rate": rate(single_period_run.redemption_rate),
    "cash_used": money(single_period_result.cash_used),
    "gross_sales": money(gross_sales(single_period_result)),
    "post_haircut_cash_raised": money(single_period_result.total_post_haircut_cash_raised),
    "dilution_amount": money(single_period_result.dilution_amount),
    "dilution_rate": rate(single_period_result.dilution_rate),
    "shortfall": money(single_period_result.shortfall),
    "remaining_liquid_buffer_rate": rate(single_period_result.remaining_liquid_buffer_rate),
    "minimum_cash_buffer_preserved": yes_no(single_period_result.minimum_cash_buffer_preserved),
}

print_profile(liquidation_summary)

In [ ]:
pd.DataFrame(
    [
        {
            "position_id": asset.position_id,
            "asset_group": asset.asset_group.value,
            "gross_sale_amount": money(asset.gross_sale_amount),
            "post_haircut_cash_raised": money(asset.post_haircut_cash_raised),
            "haircut_cost": money(asset.haircut_cost),
        }
        for asset in single_period_result.assets_liquidated
    ]
)

## Activation Assessment

The application labels LMT outputs as simulated activation assessment. These diagnostics support calibration review; they do not decide whether a fund manager activates an LMT.

In [ ]:
activation = single_period_run.lmt_activation
activation_summary = {
    "swing_pricing": yes_no(activation.swing_activated),
    "redemption_gate": yes_no(activation.gate_activated),
    "liquidity_buffer_breach": yes_no(activation.buffer_breached),
    "calibration_adequacy": activation.calibration_adequacy,
    "nav_after_redemption_before_lmt": money(activation.nav_after_redemption_before_lmt),
    "applied_swing_factor_rate": rate(activation.applied_swing_factor_rate),
    "applied_cost_recovery_amount": money(activation.applied_cost_recovery_amount),
    "redemption_paid_amount": money(activation.redemption_paid_amount),
    "redemption_deferred_amount": money(activation.redemption_deferred_amount),
    "current_post_lmt_nav": money(activation.current_post_lmt_nav),
    "remaining_liquid_buffer_rate": rate(activation.remaining_liquid_buffer_rate),
}

print_profile(activation_summary)

## Scenario Matrix View

The matrix view runs the selected redemption and liquidation strategy across the available market conditions. This is the same comparison boundary used by the application: each column is an independent service run.

In [ ]:
matrix_runs = run_scenario_across_market_conditions(
    inputs,
    fund_id=selected_fund_id,
    strategy_id=selected_strategy_id,
    redemption_scenario_id_override=selected_redemption_scenario_id,
)

matrix_rows = []
for run in matrix_runs:
    outcome = build_scenario_matrix_outcome(run)
    matrix_rows.append(
        {
            "market_stress": run.market_stress.name,
            "current_pre_lmt_nav": money(run.current_pre_lmt_nav),
            "redemption_amount": money(run.redemption_amount),
            "realised_liquidity_cost": money(outcome.realised_liquidity_cost),
            "current_post_lmt_nav": money(outcome.current_post_lmt_nav),
            "remaining_liquid_buffer_rate_after_lmt": rate(
                outcome.remaining_liquid_buffer_rate_after_lmt
            ),
            "calibration_adequacy": run.lmt_activation.calibration_adequacy,
        }
    )

pd.DataFrame(matrix_rows)

## Historical Market Stress Reference Rows

Historical market stress rows are reference context. They are loaded by the same application service and displayed as context rather than silently changing the active workflow.

In [ ]:
pd.DataFrame(build_historical_result_rows(inputs, single_period_run))

# Part 2 - 12-Month Redemption Path Workflow

The 12-month path uses the implemented multi-period service. Normal months draw investor-class redemption rates from the Beta methodology. Selected redemption-stress months replace the sampled normal rate with each class's stress redemption rate. Deferred backlog is carried forward by investor class, and liquidation is calculated only for paid redemption.

In [ ]:
path_run = run_sample_redemption_path(
    inputs,
    fund_id=selected_fund_id,
    strategy_id=selected_strategy_id,
    redemption_scenario_id=selected_redemption_scenario_id,
    lmt_parameters_override=None,
    stress_months=(1,),
    random_seed=42,
    market_stress_id=None,
    market_stress_month=None,
    behavioural_feedback_enabled=False,
    behavioural_feedback_multiplier=Decimal("1"),
)

pd.DataFrame(path_run.configuration_rows)

## Monthly Redemption Path Summary

The monthly summary shows the main state variables carried through the path: new demand, effective demand, paid redemption, deferred redemption, backlog, NAV, cash, liquid resources, and activation assessment outcome.

In [ ]:
pd.DataFrame(
    [
        {
            "month": row["month"],
            "new_redemption_demand": money(row["new_redemption_demand"]),
            "effective_redemption_demand": money(row["effective_redemption_demand"]),
            "paid_redemption": money(row["paid_redemption"]),
            "deferred_redemption": money(row["deferred_redemption"]),
            "cumulative_backlog": money(row["cumulative_backlog"]),
            "closing_nav": money(row["closing_nav"]),
            "closing_cash": money(row["closing_cash"]),
            "liquid_resources": money(row["remaining_liquid_resources"]),
            "priority_outcome": row["priority_outcome"],
        }
        for row in path_run.monthly_rows
    ]
)

## Investor-Class Demand And Backlog

Investor-class rows help reviewers see which class drives demand and how paid and deferred amounts are allocated. Backlog is not multiplied again by behavioural feedback; it is carried mechanically until paid or deferred again. Market contagion is not implemented in this workflow.

In [ ]:
pd.DataFrame(
    [
        {
            "month": row["month"],
            "client_class": row["client_class"],
            "new_redemption_demand": money(row["new_redemption_demand"]),
            "opening_backlog": money(row["opening_backlog"]),
            "effective_redemption_demand": money(row["effective_redemption_demand"]),
            "paid_redemption": money(row["paid_redemption"]),
            "deferred_redemption": money(row["deferred_redemption"]),
            "redemption_rate": rate(row["redemption_rate"]),
            "behaviour_source_outcome": row["behaviour_source_outcome"],
        }
        for row in path_run.investor_rows
    ]
)

## Monthly Activation Assessment

The path records swing-pricing, redemption-gate, liquidity-buffer, and suspension flags every month. Suspension is preserved as a methodology boundary; no new suspension methodology is introduced by the path workflow.

In [ ]:
pd.DataFrame(
    [
        {
            "month": row["month"],
            "swing_pricing": row["swing_pricing"],
            "redemption_gate": row["redemption_gate"],
            "liquidity_buffer_breach": row["liquidity_buffer_breach"],
            "suspension": row["suspension"],
            "priority_outcome": row["priority_outcome"],
            "paid_redemption": money(row["paid_redemption"]),
            "deferred_redemption": money(row["deferred_redemption"]),
            "swing_recovery": money(row["swing_recovery"]),
        }
        for row in path_run.lmt_timeline_rows
    ]
)

## Reviewer Notes

This walkthrough intentionally calls the same service layer as the application. If a service output looks wrong, the fix belongs in the domain, engine, service, data, or methodology documentation rather than in notebook-local helper code.

The separate liquidation strategy inspection notebook remains available as a low-level diagnostic aid, but it still contains local inspection helpers and should be reviewed separately if the project wants every notebook to use only application services.